# Are Returns Normal? QQ Plots and Tail Checks

Many tools in finance assume normal returns: parametric VaR, the $\sqrt{T}$ scaling of quantiles, the confidence intervals of
module 06, Black–Scholes. Before using them, check the data. A QQ plot compares the sorted returns with the quantiles a normal
distribution would produce; departures show up as bends at the ends. Formal tests (Jarque–Bera, Shapiro–Wilk) put a p-value on the
question, and tail checks count how often losses exceed what the normal model predicts. This notebook builds a QQ plot by hand, reads
its shapes, runs the tests, and explains why with daily data the answer is always "no", while with a few years of monthly data it is often
"can't tell".

## 0. Math link

- [QQ plots](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/16_qq_plots.ipynb):
  sample quantiles against theoretical quantiles.
- [Quantiles](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/12_quantiles.ipynb):
  the value below which a given share of the distribution lies, $F^{-1}(p)$.
- Skewness and kurtosis from [02/03](./03_skewness_kurtosis_fat_tails.ipynb), which the Jarque–Bera test combines.

## 1. Introduction

A risk manager computes the 99% one-day VaR of a Swiss equity portfolio as 2.33 standard deviations. Is that safe? Only if the lower tail
of the return distribution behaves like the normal's. A QQ plot answers this in one picture, and a count of past exceedances answers it
in one number.

After this notebook you can:

- build a QQ plot from scratch and recognize the shapes of fat tails, thin tails and skewness;
- add a simulation band to see which departures are larger than sampling noise;
- compute and interpret the Jarque–Bera statistic, and know its power at different sample sizes;
- check tails directly: VaR exceedances against the normal prediction, and a tail index estimate;
- compare the fit of a normal and a Student-$t$ distribution.

## 2. Theory

### The QQ plot

Sort the standardized returns $z_{(1)} \le z_{(2)} \le \dots \le z_{(n)}$. The $i$-th smallest of $n$ draws from a distribution $F$ should
lie near the quantile at probability $p_i = (i - 0.5)/n$. Plot

$$
\big(F^{-1}(p_i),\; z_{(i)}\big), \qquad i = 1, \dots, n .
$$

If the data come from $F$ (up to location and scale), the points lie on a straight line. With $F = \Phi$, the standard normal:

| shape of the QQ plot | meaning |
|---|---|
| straight line | normal |
| S-shape: low end below the line, high end above | fat tails (excess kurtosis > 0) |
| inverted S: ends flatter than the line | thin tails |
| both ends bend downwards | negative skewness (long left tail) |
| both ends bend upwards | positive skewness |

The ends of the plot rest on few observations each, so some wiggle is normal even for normal data; a simulation band (the range of QQ
plots of many normal samples of the same size) shows how much.

### The Jarque–Bera test

Under normality the sample skewness $S$ and excess kurtosis $K$ are approximately independent normal with variances $6/n$ and $24/n$
(02/03). Their standardized squares add to

$$
JB = \frac{n}{6}\Big(S^2 + \frac{K^2}{4}\Big) \;\approx\; \chi^2_2 \quad\text{under normality} .
$$

Reject normality at 5% if $JB > 5.99$. Since $JB$ grows with $n$ for any fixed non-normal shape, large samples reject even small
departures, and small samples miss large ones.

### Other tests

- **Shapiro–Wilk** compares the ordered sample with expected normal order statistics; it is the most powerful general test for moderate $n$.
- **Kolmogorov–Smirnov** uses the largest gap between the empirical and the normal CDF; with estimated mean and variance its standard
  p-values are wrong (Lilliefors correction), and it is weak in the tails, where finance cares most.

### Tail checks

- **Exceedances.** If returns are normal, the loss exceeds the 99% normal VaR on 1% of days; count the actual share. A binomial test
  says whether the difference is significant (the basis of the Basel "traffic light" backtest).
- **Tail index.** Fat tails often decay like a power law, $P(X < -x) \approx C x^{-\alpha}$. The **Hill estimator** from the $k$ largest losses
  $L_{(1)} \ge \dots \ge L_{(k+1)}$ is
  $$
  \hat\alpha = \Big(\frac1k\sum_{i=1}^{k} \ln\frac{L_{(i)}}{L_{(k+1)}}\Big)^{-1} .
  $$
  For a Student-$t$ with $\nu$ degrees of freedom, $\alpha = \nu$; moments of order $\ge \alpha$ are infinite. Daily equity returns typically
  give $\alpha$ between 3 and 5. The normal has no power-law tail ($\hat\alpha$ grows without bound as $k$ shrinks).

## 3. Worked Examples

### Example 1: plotting positions

With $n = 5$ standardized returns $-2.1, -0.4, 0.1, 0.5, 1.9$: $p_i = 0.1, 0.3, 0.5, 0.7, 0.9$ and the normal quantiles are
$-1.28, -0.52, 0, 0.52, 1.28$. The extremes $-2.1$ and $1.9$ lie well beyond $\mp 1.28$: an S-shape, but with five points it means little.

### Example 2: Jarque–Bera for daily data

Ten years of daily returns ($n = 2{,}520$) with skewness $-0.3$ and excess kurtosis 5: $JB = 420 \times (0.09 + 6.25) = 2{,}663$, far beyond 5.99.
Normality is rejected with overwhelming confidence.

### Example 3: Jarque–Bera for monthly data

Five years of monthly returns ($n = 60$) with skewness $-0.3$ and excess kurtosis 1: $JB = 10 \times (0.09 + 0.25) = 3.4$, below 5.99. Normality is
not rejected, though the data are not normal.

### Example 4: exceedances

Over 1,000 days, a normal 99% VaR should be exceeded about 10 times (binomial standard deviation 3.1). Observing 22 exceedances is almost four
standard deviations too many: the normal model underestimates the tail.

## 4. Finance Application

### Where it matters

- **VaR backtesting.** Regulators count exceedances of the 99% VaR over 250 days; too many push a bank into the yellow or red zone of the
  Basel traffic light and raise its capital multiplier.
- **Model choice.** QQ plots against the normal and the Student-$t$ decide which distribution goes into a parametric VaR or a Monte Carlo
  simulation.
- **Performance statistics.** Sharpe-ratio confidence intervals and $t$-tests of alpha (module 06) rely on approximate normality of
  averages; strong skewness or kurtosis widens the true intervals.
- **Option pricing.** The departure of the QQ plot from a line at the left end is the same fact as the volatility skew in option prices.

### The usual misreadings

- "The test did not reject, so returns are normal." With 60 monthly observations it rarely can; absence of evidence is not evidence.
- "The test rejected, so the normal model is useless." With 10,000 daily observations any tiny deviation is rejected; the question is
  whether the deviation matters for the decision.
- "The QQ plot is straight in the middle, so the model is fine." Risk lives in the ends of the plot.
- "Monthly returns pass the test, so daily VaR can assume normality." Tails fade with aggregation (02/03); check at the horizon you use.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### Three samples

1. **Normal**: ten years of daily returns from a normal distribution (a benchmark that should pass).
2. **Daily equity**: ten years of daily returns with volatility regimes and occasional crashes, as in 02/03.
3. **Monthly equity**: five years of monthly sums of the same kind of process.

Returns are decimals; 252 trading days a year.

In [ ]:
def regime_returns(n_days, rng):
    regime = np.zeros(n_days, dtype=int)
    u = rng.uniform(size=n_days)
    for t in range(1, n_days):
        stay = 0.995 if regime[t - 1] == 0 else 0.96
        regime[t] = regime[t - 1] if u[t] < stay else 1 - regime[t - 1]
    vol = np.where(regime == 0, 0.008, 0.025)
    jumps = np.where(rng.uniform(size=n_days) < 1 / 500, rng.normal(-0.05, 0.02, n_days), 0.0)
    return 0.0004 + vol * rng.standard_normal(n_days) + jumps

n_daily = 10 * 252
normal_daily = rng.normal(0.0004, 0.012, n_daily)
equity_daily = regime_returns(n_daily, rng)
equity_monthly = regime_returns(60 * 21, rng).reshape(60, 21).sum(axis=1)

samples = {"normal daily (n = 2520)": normal_daily, "equity daily (n = 2520)": equity_daily,
           "equity monthly (n = 60)": equity_monthly}

### A QQ plot by hand

In [ ]:
def qq_points(x, dist=stats.norm):
    z = np.sort((x - x.mean()) / x.std(ddof=1))
    n = len(z)
    p = (np.arange(1, n + 1) - 0.5) / n
    return dist.ppf(p), z

theo, emp = qq_points(equity_daily)
osm, osr = stats.probplot((equity_daily - equity_daily.mean()) / equity_daily.std(ddof=1), dist="norm", fit=False)
print("five lowest points (theoretical, empirical):")
print(np.round(np.column_stack([theo[:5], emp[:5]]), 2))
print("max difference to scipy.stats.probplot positions:", np.abs(np.sort(osm) - theo).max().round(4))

The lowest standardized daily return lies about 8 standard deviations below the mean, where a normal sample of this size would put its
minimum near −3.5. SciPy's `probplot` uses slightly different plotting positions (Filliben's), which differ from $(i - 0.5)/n$ only at the
extremes.

### Tests

In [ ]:
def jarque_bera(x):
    z = (x - x.mean()) / x.std()
    S, K = np.mean(z**3), np.mean(z**4) - 3
    jb = len(x) / 6 * (S**2 + K**2 / 4)
    return S, K, jb, stats.chi2.sf(jb, 2)

rows = []
for name, x in samples.items():
    S, K, jb, p = jarque_bera(x)
    rows.append({"sample": name, "skewness": S, "excess kurtosis": K, "JB": jb, "JB p-value": p,
                 "scipy JB": stats.jarque_bera(x).statistic, "Shapiro-Wilk p-value": stats.shapiro(x).pvalue})
pd.DataFrame(rows).set_index("sample")

The hand-computed Jarque–Bera statistic matches SciPy's. The normal sample passes both tests. The daily equity sample fails them by an
enormous margin. Whether the five years of monthly equity returns pass depends on the particular draw: the table shows this sample's
verdict, and the power study below shows how often such a sample is caught.

### Power: how often does Jarque–Bera detect fat tails?

In [ ]:
def rejection_rate(draw, n, reps=2000):
    return np.mean([jarque_bera(draw(n))[3] < 0.05 for _ in range(reps)])

rows = []
for n in [24, 60, 120, 250, 1000]:
    rows.append({"n": n,
                 "normal (size)": rejection_rate(lambda m: rng.normal(size=m), n),
                 "t(5)": rejection_rate(lambda m: rng.standard_t(5, m), n),
                 "t(10)": rejection_rate(lambda m: rng.standard_t(10, m), n)})
power = pd.DataFrame(rows).set_index("n")
power

For normal data the test should reject 5% of the time; with small samples it rejects less often (the $\chi^2$ approximation is poor there).
For $t(5)$ data, which have clearly fat tails, the test catches the departure in only about half of five-year monthly samples ($n = 60$);
for the milder $t(10)$ it usually fails until hundreds of observations are available.

### Tail checks: VaR exceedances

In [ ]:
rows = []
for name, x in [("normal daily", normal_daily), ("equity daily", equity_daily)]:
    for level in [0.95, 0.99, 0.995]:
        var_normal = x.mean() + stats.norm.ppf(1 - level) * x.std(ddof=1)        # a negative return
        exceed = int(np.sum(x < var_normal))
        expected = (1 - level) * len(x)
        p_value = stats.binomtest(exceed, len(x), 1 - level).pvalue
        rows.append({"sample": name, "VaR level": level, "normal VaR (return)": var_normal,
                     "exceedances": exceed, "expected": expected, "binomial p-value": p_value})
pd.DataFrame(rows).set_index(["sample", "VaR level"])

For the normal sample the exceedance counts are close to their expectations. For the equity sample the 95% VaR is exceeded *less*
often than expected (the distribution is peaked, and the volatility is inflated by the tails), while the 99% and 99.5% VaR are exceeded
far more often. A normal VaR calibrated to the right volatility is too conservative at moderate levels and too optimistic at extreme ones.

### Tail index and a Student-t fit

In [ ]:
def hill(losses, k):
    L = np.sort(losses[losses > 0])[::-1]
    return 1 / np.mean(np.log(L[:k] / L[k]))

losses_eq = -(equity_daily - equity_daily.mean())
losses_n = -(normal_daily - normal_daily.mean())
t5 = rng.standard_t(5, n_daily)
for k in [25, 50, 100, 200]:
    print(f"k = {k:3d}: Hill alpha  equity {hill(losses_eq, k):5.2f}, normal {hill(losses_n, k):5.2f}, t(5) {hill(-t5, k):5.2f}")

df_fit, loc_fit, scale_fit = stats.t.fit(equity_daily)
print(f"\nStudent-t fit to equity daily returns: nu = {df_fit:.2f}, loc = {loc_fit:.5f}, scale = {scale_fit:.5f}")

The Hill estimates for the equity sample range from about 2 to 4 and fall as $k$ grows, because the largest losses come from the crash
days and the turbulent regime, two different tails mixed together. For the $t(5)$ sample they are stable around 3.1–3.5, below the true
$\alpha = 5$: the Hill estimator is biased when the power law holds only far out. For the normal sample $\hat\alpha$ is larger and keeps
rising as $k$ shrinks: no power-law tail. The fitted Student-$t$ has 2.7 degrees of freedom, a very heavy tail.

## 6. Visualizations

In [ ]:
def simulation_band(n, reps=500, q=(0.025, 0.975)):
    sims = np.sort(rng.standard_normal((reps, n)), axis=1)
    sims = (sims - sims.mean(axis=1, keepdims=True)) / sims.std(axis=1, ddof=1, keepdims=True)
    return np.quantile(sims, q[0], axis=0), np.quantile(sims, q[1], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, x) in zip(axes, samples.items()):
    theo, emp = qq_points(x)
    lo, hi = simulation_band(len(x))
    ax.fill_between(theo, lo, hi, color="grey", alpha=0.3, label="95% band for normal samples")
    ax.plot(theo, emp, ".", ms=3)
    lim = max(4, np.abs(emp).max() * 1.05)
    ax.plot([-lim, lim], [-lim, lim], "k", lw=0.8)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-lim, lim)
    ax.set_title(name)
    ax.set_xlabel("normal quantile")
    ax.set_ylabel("standardized sample quantile")
    ax.grid(True, alpha=0.3)
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

Left: the normal sample stays inside the band. Middle: the daily equity sample shows the S-shape of fat tails, with the left end far
below the band (crashes) and the right end above it; even the centre is flatter than the line. Right: with 60 monthly observations the band
is wide, and departures, if any, are hard to separate from noise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, dist) in zip(axes, [("against the normal", stats.norm),
                                    (f"against the fitted t({df_fit:.1f})", stats.t(df_fit))]):
    if dist is stats.norm:
        theo, emp = qq_points(equity_daily)
    else:
        emp = np.sort((equity_daily - loc_fit) / scale_fit)
        theo = dist.ppf((np.arange(1, n_daily + 1) - 0.5) / n_daily)
    ax.plot(theo, emp, ".", ms=3)
    lim = np.abs(np.concatenate([theo, emp])).max() * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k", lw=0.8)
    ax.set_title(f"Equity daily returns {title}")
    ax.set_xlabel("theoretical quantile")
    ax.set_ylabel("sample quantile (standardized)")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Against the normal the points bend away at both ends. Against the fitted $t(2.7)$ they follow the line over a much wider range, from
about −7 to +6 scale units. Beyond that the sample is lighter-tailed than the fitted $t$: with so few degrees of freedom the $t$ predicts
more extreme days than occurred, the price of fitting the body and the crash days with one symmetric shape. The left end still reaches
further than the right, which a symmetric $t$ cannot capture.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for col in power.columns:
    ax.semilogx(power.index, power[col], "o-", label=col)
ax.axhline(0.05, color="grey", ls=":", label="5% significance level")
ax.set_title("Jarque–Bera: share of samples rejected at 5%")
ax.set_xlabel("sample size n (log scale)")
ax.set_ylabel("rejection rate")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

Power grows with the sample size and with the strength of the departure. Two years of monthly data (24 points) rarely reject normality,
even for $t(5)$ data (20% of samples); four years of daily data (1,000 points) almost always do for $t(5)$ and $t(10)$.

## 7. Exercises

### Basic

1. For $n = 10$, compute the plotting positions $p_i = (i - 0.5)/n$ and the corresponding standard normal quantiles.
2. A sample of 500 daily returns has skewness $-0.4$ and excess kurtosis 3. Compute the Jarque–Bera statistic and its p-value.
3. A pension fund's consultant tests five years of monthly returns of a hedge fund with Jarque–Bera and reports "p = 0.21, returns are normal,
   so the normal VaR is appropriate". **Interpret** this conclusion. What would you check instead?

### Intermediate

4. Backtest a normal 99% VaR on the daily equity sample over rolling 250-day windows: estimate the mean and volatility on each window, count
   exceedances on the following day, and compare the total with the expected number. Classify the result with the Basel traffic light per
   250 days (green: 0–4, yellow: 5–9, red: 10 or more exceedances).
5. Draw 60 monthly returns from the put-selling strategy of 02/03 (or any strongly skewed distribution) and make a QQ plot. Which feature of
   the plot reveals the skewness, and does Jarque–Bera detect it?

### Advanced

6. Show that if $X$ has a Student-$t$ distribution with $\nu$ degrees of freedom, $P(X > x) \sim c\,x^{-\nu}$ as $x \to \infty$, so the tail index is
   $\nu$. Why does this make $E[|X|^k]$ infinite for $k \ge \nu$?
7. Explain why the QQ plot of standardized data uses the sample mean and standard deviation, and why a single extreme observation can make the
   whole central part of the plot look flatter than the line (as in the daily equity panel).

## 8. Solutions / Checks

In [ ]:
# 1.
p10 = (np.arange(1, 11) - 0.5) / 10
print("1.", np.round(p10, 2), np.round(stats.norm.ppf(p10), 3))

# 2.
jb2 = 500 / 6 * (0.4**2 + 3**2 / 4)
print(f"2. JB = {jb2:.1f}, p-value {stats.chi2.sf(jb2, 2):.2e}")

# 4.
window = 250
exceed = []
for t in range(window, n_daily):
    hist = equity_daily[t - window:t]
    var99 = hist.mean() + stats.norm.ppf(0.01) * hist.std(ddof=1)
    exceed.append(equity_daily[t] < var99)
exceed = np.array(exceed)
print(f"4. exceedances {exceed.sum()} over {len(exceed)} days, expected {0.01 * len(exceed):.1f}")
per_block = exceed[: len(exceed) // 250 * 250].reshape(-1, 250).sum(axis=1)
zones = ["green" if k <= 4 else "yellow" if k <= 9 else "red" for k in per_block]
print("   per 250-day block:", list(per_block), zones)

**3.** With 60 observations Jarque–Bera has low power (the table above: about half of $t(5)$ samples are missed), and hedge-fund returns are often
skewed in ways a few years of data do not reveal. A high p-value means "no evidence against normality", not "normal". Check the worst months, the
QQ plot, drawdowns, the strategy's exposure to options or credit events, and stress scenarios rather than relying on the test.

**4.** Rolling normal VaR adapts to the volatility regimes, which helps, but exceedances (43) are still almost twice the expected 22.7; five of the nine
250-day blocks fall in the yellow zone, none in the red. Under the Basel rules the yellow zone raises the capital multiplier.

In [ ]:
# 5. a strongly skewed sample: +0.5% most months, occasional large losses
skewed = np.where(rng.uniform(size=60) < 0.92, rng.normal(0.005, 0.002, 60), rng.normal(-0.05, 0.03, 60))
theo5, emp5 = qq_points(skewed)
S5, K5, jb5, p5 = jarque_bera(skewed)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(theo5, emp5, "o", ms=4)
ax.plot([-3, 3], [-3, 3], "k", lw=0.8)
ax.set_title(f"Skewed sample, n = 60: JB p-value {p5:.3g}")
ax.set_xlabel("normal quantile")
ax.set_ylabel("standardized sample quantile")
ax.grid(True, alpha=0.3)
plt.show()
print(f"5. skewness {S5:.2f}, excess kurtosis {K5:.2f}")

**5.** The left end of the plot drops far below the line, while the right end lies below it as well (the gains are capped): both ends bend
downwards, the signature of negative skewness. With a skew this strong, Jarque–Bera detects it even with 60 observations.

**6.** The $t$ density is $f(x) \propto (1 + x^2/\nu)^{-(\nu+1)/2} \sim c'\,|x|^{-(\nu+1)}$ for large $|x|$; integrating from $x$ to $\infty$ gives
$P(X > x) \sim c\,x^{-\nu}$. Then $E[|X|^k] = \int |x|^k f(x)\,dx$ behaves like $\int^\infty x^{k - \nu - 1}dx$, which diverges for $k \ge \nu$.

**7.** Standardizing removes location and scale so that the line $y = x$ is the reference. An extreme observation inflates the sample standard
deviation; dividing by it shrinks every other standardized value, so the bulk of the data falls inside the line (a flatter centre) while the
extreme points still stick out.

## 9. Conclusion

- A QQ plot of standardized returns against normal quantiles is a straight line for normal data; S-shapes mean fat tails, bending ends mean skewness.
- Simulation bands show which departures are larger than sampling noise, which matters most for small samples.
- Jarque–Bera combines skewness and kurtosis into a $\chi^2_2$ statistic; it rejects everything with daily data and little with a few years of
  monthly data.
- Tail checks (VaR exceedances, Hill tail index) test what risk management actually depends on; daily equity tails typically have an index around 3–5.
- A Student-$t$ often fits the body and both tails much better than the normal; asymmetric crash tails need more.

This completes module 02. Next: [03/01 Variance of a sum and a difference](../03_covariance_and_two_assets/01_variance_of_sum_and_difference.ipynb).